# Sprints AI-Powered Learner Success & Support Agent
## Notebook 02: PyTorch Machine Learning & Deep Sequence Modeling (Sprints 2 & 3)
**Client:** Sprints (sprints.ai)  
**Track:** AI Pioneers Nationwide Cohort  
**Author:** Mohammed Wael (Lead AI Engineer)

---

### Notebook Overview
This notebook integrates **Sprint 2 (Intro to Machine Learning)** and **Sprint 3 (Deep Learning)**:
1. **Methodological Rigor & Zero Data Leakage**:
   - Clean 3-way split: 70% Train ($N=1,400$), 15% Validation ($N=300$), 15% Test ($N=300$).
   - Imputation and scalers fitted strictly on Train set.
   - Target variable `project_score` is NOT synthetically imputed based on `dropped_out`.
2. **PyTorch Supervised Risk Classifier (Sprint 2)**:
   - Evaluates both **Experiment A (Early Risk Weeks 1-3)** and **Experiment B (Full Trajectory Weeks 1-5)**.
   - Exceeds acceptance criteria: **Test $F_1 \ge 0.82$**, achieving 0 False Negatives on High Risk students.
3. **PyTorch Sequence LSTM (Sprint 3)**:
   - Deep recurrent architecture processing $[q_1..q_5]$ trajectory.
   - Mathematical formulation distinguishing **Forget Gate ($f_t$)**, **Cell State ($C_t$ - Constant Error Carousel)**, and **Hidden State ($h_t$)**.
   - Forecasts capstone project outcome strictly on non-null submissions ($N=1,639$).
4. **Latent Space PCA Clustering**:
   - Visualizes penultimate layer embeddings separating student risk clusters.

In [1]:
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, r2_score, mean_squared_error, mean_absolute_error
from sklearn.decomposition import PCA

sys.path.append(os.path.join('..', 'scripts'))
from train_pytorch import (
    load_and_split_data,
    LearnerRiskClassifier,
    LearnerSequenceLSTM
)

print(f"PyTorch Version: {torch.__version__}")

PyTorch Version: 2.14.1+cpu


### 1. Data Ingestion & Leakage-Free Preprocessing

In [2]:
csv_path = os.path.join('..', 'data', 'sprints_learner_data.csv')
train_df, val_df, test_df = load_and_split_data(csv_path)

print(f"Train set: {len(train_df)} (70%) | Val set: {len(val_df)} (15%) | Test set: {len(test_df)} (15%)")
print(f"Test Set Dropped count: {test_df['dropped_out'].sum()} ({test_df['dropped_out'].mean()*100:.1f}%)")
print(f"Test Set Non-null project scores: {test_df['project_score'].notna().sum()} / {len(test_df)}")

Loaded raw dataset from ..\data\sprints_learner_data.csv: 2000 rows, 10 columns.
Dataset Split Sizes -> Train: 1400 (70%), Val: 300 (15%), Test: 300 (15%)
Leak-free preprocessing complete. Training statistics strictly isolated.
Train set: 1400 (70%) | Val set: 300 (15%) | Test set: 300 (15%)
Test Set Dropped count: 69 (23.0%)
Test Set Non-null project scores: 248 / 300


### 2. Sprint 2: PyTorch Supervised Feed-Forward Neural Network (Risk Classifier)
Evaluating the saved checkpoint `models/risk_classifier.pth` on the independent Test set ($N=300$).

In [3]:
ckpt_path = os.path.join('..', 'models', 'risk_classifier.pth')
ckpt = torch.load(ckpt_path, weights_only=False)

feature_cols = ckpt['feature_cols']
print("Loaded Features:", feature_cols)

X_test = test_df[feature_cols].values
y_test = test_df['dropped_out'].values

# Scale using train-fitted scaler parameters
X_test_scaled = (X_test - ckpt['scaler_mean']) / ckpt['scaler_scale']

model = LearnerRiskClassifier(input_dim=len(feature_cols), hidden_dim=64)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

with torch.no_grad():
    logits = model(torch.tensor(X_test_scaled, dtype=torch.float32))
    probs = torch.sigmoid(logits).squeeze().numpy()

threshold = ckpt.get('decision_threshold', 0.42)
preds = (probs >= threshold).astype(int)

print("\n" + "="*50)
print("EXPERIMENT B (FULL TRAJECTORY WEEKS 1-5) INDEPENDENT TEST REPORT:")
print(classification_report(y_test, preds, target_names=['Low Risk', 'High Risk'], digits=4))
print(f"Test ROC-AUC: {roc_auc_score(y_test, probs):.4f}")
print("="*50)

Loaded Features: ['week_1_quiz', 'week_2_quiz', 'week_3_quiz', 'week_4_quiz', 'week_5_quiz', 'score_delta_w1_w5', 'quiz_mean', 'quiz_std', 'quiz_min', 'recent_performance_ratio', 'missing_quiz_count']

EXPERIMENT B (FULL TRAJECTORY WEEKS 1-5) INDEPENDENT TEST REPORT:
              precision    recall  f1-score   support

    Low Risk     1.0000    1.0000    1.0000       231
   High Risk     1.0000    1.0000    1.0000        69

    accuracy                         1.0000       300
   macro avg     1.0000    1.0000    1.0000       300
weighted avg     1.0000    1.0000    1.0000       300

Test ROC-AUC: 1.0000


#### Experiment A: Proactive Early Risk Classifier (Weeks 1-3)
Evaluates whether students at risk of dropout can be reliably identified after only 3 weeks of curriculum telemetry.

In [4]:
early_feature_cols = ckpt['early_feature_cols']
X_test_early = test_df[early_feature_cols].values
X_test_early_scaled = (X_test_early - ckpt['early_scaler_mean']) / ckpt['early_scaler_scale']

early_model = LearnerRiskClassifier(input_dim=len(early_feature_cols), hidden_dim=64)
early_model.load_state_dict(ckpt['early_model_state_dict'])
early_model.eval()

with torch.no_grad():
    early_logits = early_model(torch.tensor(X_test_early_scaled, dtype=torch.float32))
    early_probs = torch.sigmoid(early_logits).squeeze().numpy()

early_thresh = ckpt.get('early_threshold', 0.35)
early_preds = (early_probs >= early_thresh).astype(int)

print("="*50)
print("EXPERIMENT A (WEEKS 1-3) INDEPENDENT TEST REPORT:")
print(classification_report(y_test, early_preds, target_names=['Low Risk', 'High Risk'], digits=4))
print(f"Early Test ROC-AUC: {roc_auc_score(y_test, early_probs):.4f}")
print("="*50)

EXPERIMENT A (WEEKS 1-3) INDEPENDENT TEST REPORT:
              precision    recall  f1-score   support

    Low Risk     0.9956    0.9870    0.9913       231
   High Risk     0.9577    0.9855    0.9714        69

    accuracy                         0.9867       300
   macro avg     0.9767    0.9863    0.9814       300
weighted avg     0.9869    0.9867    0.9867       300

Early Test ROC-AUC: 0.9996


#### Confusion Matrix & Latent Space Projections

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Low Risk', 'High Risk'], yticklabels=['Low Risk', 'High Risk'])
axes[0].set_title(f'Test Confusion Matrix (Threshold={threshold:.2f})', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Latent Space (Penultimate Layer)
with torch.no_grad():
    latent_reps = model.extract_latent_features(torch.tensor(X_test_scaled, dtype=torch.float32)).numpy()

pca = PCA(n_components=2)
latent_pca = pca.fit_transform(latent_reps)

scatter = axes[1].scatter(latent_pca[:, 0], latent_pca[:, 1], c=y_test, cmap='coolwarm', alpha=0.8, edgecolors='k')
axes[1].set_title('Latent Space Distribution (Behavioral Clusters)', fontsize=12, fontweight='bold')
axes[1].set_xlabel(f'Latent PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'Latent PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
fig.colorbar(scatter, ax=axes[1], label='True Dropout Status')

plt.tight_layout()
plt.show()

C:\Users\mw851\AppData\Local\Temp\ipykernel_17192\332020224.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3. Sprint 3: Deep Sequence LSTM Trajectory Forecasting
#### Architectural Distinction: Forget Gate vs Cell State vs Hidden State
$$\begin{aligned}
f_t &= \sigma(W_f \cdot [h_{t-1}, x_t] + b_f) \quad &\text{(Forget Gate: adaptive dampening of early-week noise)} \\
i_t &= \sigma(W_i \cdot [h_{t-1}, x_t] + b_i) \quad &\text{(Input Gate: select candidate evidence)} \\
\tilde{C}_t &= \tanh(W_c \cdot [h_{t-1}, x_t] + b_c) \quad &\text{(Candidate Memory Vector)} \\
C_t &= f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \quad &\text{(Cell State: Constant Error Carousel preventing vanishing gradients)} \\
o_t &= \sigma(W_o \cdot [h_{t-1}, x_t] + b_o) \quad &\text{(Output Gate)} \\
h_t &= o_t \odot \tanh(C_t) \quad &\text{(Hidden State: projected trajectory encoding)}
\end{aligned}$$

The model is trained and evaluated strictly on learners with real capstone project submissions ($N=1,639$), avoiding synthetic label leakage.

In [6]:
lstm_ckpt_path = os.path.join('..', 'models', 'sequence_predictor.pth')
ckpt_lstm = torch.load(lstm_ckpt_path, weights_only=False)

lstm_model = LearnerSequenceLSTM(input_size=1, hidden_size=ckpt_lstm['hidden_size'], num_layers=ckpt_lstm['num_layers'])
lstm_model.load_state_dict(ckpt_lstm['model_state_dict'])
lstm_model.eval()

# Filter strictly to learners with true project submissions in the test set
test_lstm_df = test_df[test_df['project_score'].notna()].copy()
seq_cols = ['week_1_quiz', 'week_2_quiz', 'week_3_quiz', 'week_4_quiz', 'week_5_quiz']
X_te_s = test_lstm_df[seq_cols].values / 100.0
y_te_s = test_lstm_df['project_score'].values

with torch.no_grad():
    tensor_seq = torch.tensor(X_te_s, dtype=torch.float32).unsqueeze(-1)
    preds_norm, _ = lstm_model(tensor_seq)
    predicted_projects = preds_norm.squeeze().numpy() * 100.0

r2 = r2_score(y_te_s, predicted_projects)
rmse = np.sqrt(mean_squared_error(y_te_s, predicted_projects))
mae = mean_absolute_error(y_te_s, predicted_projects)

print(f"Sequence LSTM Test RMSE: {rmse:.2f} points")
print(f"Sequence LSTM Test MAE:  {mae:.2f} points")
print(f"Sequence LSTM Test R^2:  {r2:.4f}")

Sequence LSTM Test RMSE: 7.63 points


Sequence LSTM Test MAE:  5.59 points
Sequence LSTM Test R^2:  0.8229


In [7]:
# Trajectory Visualization
plt.figure(figsize=(10, 5))
sample_indices = [2, 9, 14, 25]
weeks = [1, 2, 3, 4, 5]

for idx in sample_indices:
    p = plt.plot(weeks, X_te_s[idx] * 100.0, marker='o', linewidth=2, label=f'Learner #{idx} Weekly Quizzes')
    col = p[0].get_color()
    plt.scatter(6, predicted_projects[idx], marker='*', s=180, color=col, label=f'Pred Project: {predicted_projects[idx]:.1f}')
    plt.scatter(6, y_te_s[idx], marker='x', s=120, color=col)

plt.title('PyTorch LSTM Sequential Trajectory & Capstone Outcome Forecast', fontsize=13, fontweight='bold')
plt.xlabel('Bootcamp Timeline')
plt.ylabel('Score (0-100)')
plt.xticks([1, 2, 3, 4, 5, 6], ['W1 Quiz', 'W2 Quiz', 'W3 Quiz', 'W4 Quiz', 'W5 Quiz', 'Capstone Project'])
plt.legend(bbox_to_anchor=(1.04, 1), loc='upper left', fontsize=8)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

C:\Users\mw851\AppData\Local\Temp\ipykernel_17192\1321053427.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
